# Penerjemah Aceh → Indonesia: start the app (no training, about 5 minutes)
Use this after `train_colab.ipynb` has trained and saved your model once.
1. Menu **Runtime → Change runtime type → T4 GPU → Save** (a CPU also works, just slower).
2. Menu **Runtime → Run all**, paste your Hugging Face token when asked, press **Enter**.
3. The last cell prints `Running on public URL: https://xxxx.gradio.live`. Open or share that link.
   It works while this tab stays open (Colab stops after some hours, or when the tab is idle for a long time).

In [ ]:
from getpass import getpass
HF_TOKEN = getpass('Paste your Hugging Face token, then press Enter (Enter alone = use the base model): ').strip()

In [ ]:
import os
REPO = os.path.abspath('aceh_code')
if not os.path.isdir(REPO):
    !git clone -q -b claude/dataset-product-build-108xu8 https://github.com/Mutaib-yye/aceh_code {REPO}
%cd {REPO}
!git pull -q
!pip install -q "transformers==5.19.0" "sentencepiece>=0.2" "gradio==6.29.1"

In [ ]:
MODEL_ID = 'facebook/nllb-200-distilled-600M'           # base model, used if yours is not found
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN                     # lets the app download your private model
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    try:
        mine = f"{api.whoami()['name']}/ace-id-nllb"
        api.model_info(mine)
        MODEL_ID = mine
    except Exception as e:
        print('Your fine-tuned model was not found on Hugging Face, so the base model is used:', str(e).splitlines()[0])
print('model:', MODEL_ID)
import sys, importlib.util
sys.path.insert(0, os.path.join(REPO, 'space'))
spec = importlib.util.spec_from_file_location('space_app', os.path.join(REPO, 'space/app.py'))
app = importlib.util.module_from_spec(spec); spec.loader.exec_module(app)
app.start(MODEL_ID, share=True)      # prints: Running on public URL: https://xxxx.gradio.live  <- open / share this link